# A3-M3-2 · 실습 과제
01 노트북을 먼저 실행해 chroma_db/ 폴더가 있어야 합니다. 준비 셀부터 순서대로 실행합니다.

## 준비
모든 과제가 함께 쓰는 준비 셀입니다. 01 노트북을 먼저 실행해 chroma_db/가 있어야 합니다.

In [ ]:
import time
import numpy as np
import pandas as pd
import chromadb
from emb import embed, RERANK_MODEL

client = chromadb.PersistentClient(path="chroma_db")
col = client.get_collection("shop_faq")
tests = pd.read_csv("shop_faq_tests.csv")

def search(question, k=3, where=None):
    r = col.query(query_embeddings=embed([question], kind="query"), n_results=k, where=where)
    return pd.DataFrame({
        "faq_id": [int(i) for i in r["ids"][0]],
        "score": [round(1 - d, 3) for d in r["distances"][0]],
        "category": [m["category"] for m in r["metadatas"][0]],
        "question": [m["question"] for m in r["metadatas"][0]],
        "doc": r["documents"][0],
    })

def evaluate(fn, **kw):
    rows = []
    for q, gold in tests.itertuples(index=False):
        ranked = fn(q, **kw)["faq_id"].tolist()
        rows.append({"question": q, "gold": gold, "top1": ranked[0],
                     "rank": ranked.index(gold) + 1 if gold in ranked else np.nan})
    return pd.DataFrame(rows)

def summary(ev):
    return {"Hit@1": round(float((ev["rank"] == 1).mean()), 3), "Hit@3": round(float((ev["rank"] <= 3).mean()), 3),
            "MRR": round(float((1 / ev["rank"]).fillna(0).mean()), 3)}

print("FAQ", col.count(), "건 | 평가 질문", len(tests), "개")
from transformers import pipeline
rr = pipeline("text-classification", model=RERANK_MODEL, device=-1)

def search_rerank(question, k=3, pool=8, where=None):
    cand = search(question, k=pool, where=where)
    pairs = [{"text": question, "text_pair": d} for d in cand["doc"]]
    cand["rerank"] = [round(s["score"], 3) for s in rr(pairs, function_to_apply="none")]
    return cand.sort_values("rerank", ascending=False).head(k)

## 과제 1 · 카테고리 필터로 평가하기
평가 질문마다 정답 FAQ의 카테고리를 `where`로 걸고 검색해서, 필터 없을 때와 Hit@1·MRR을 비교하세요. 정답 FAQ의 카테고리는 `col.get(ids=[str(gold)])`로 꺼냅니다.

In [ ]:
# TODO

## 과제 2 · FAQ 문장 고쳐서 검색 개선하기
'비번 까먹었어요'가 1위로 찾히도록 FAQ 601의 문서에 고객이 쓰는 표현('비번')을 넣어 `upsert`하세요. 다시 평가해 Hit@1이 바뀌는지 보고, 끝나면 원래 문서로 되돌리세요.

In [ ]:
# TODO

## 과제 3 · 두 필터를 함께
'언제까지 돼요?'를 검색하되 카테고리가 `교환반품` 또는 `적립금`이고 원문에 '일'이 들어간 FAQ만 찾으세요(`where` + `where_document`).

In [ ]:
# TODO